# Chapter 6: Double Q-learning vs Conventional Q-learning

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
actions = ['left', 'right']

def sample_right_reward(rng):
    return rng.normal(-0.1, 1.0)

def q_learning(episodes=200, alpha=0.1, epsilon=0.1):
    Q = {'A': {a: 0.0 for a in actions}, 'B': {'go': 0.0}}
    left_count = 0
    for _ in range(episodes):
        a = 'left' if rng.random() < epsilon else max(actions, key=lambda x: Q['A'][x])
        if a == 'left':
            left_count += 1
            reward = 0.0 + Q['B']['go']
            Q['A']['left'] += alpha * (reward - Q['A']['left'])
            r2 = sample_right_reward(rng)
            Q['B']['go'] += alpha * (r2 - Q['B']['go'])
        else:
            r = sample_right_reward(rng)
            Q['A']['right'] += alpha * (r - Q['A']['right'])
    return left_count / episodes

def double_q_learning(episodes=200, alpha=0.1, epsilon=0.1):
    Q1 = {'A': {a: 0.0 for a in actions}, 'B': {'go': 0.0}}
    Q2 = {'A': {a: 0.0 for a in actions}, 'B': {'go': 0.0}}
    left_count = 0
    for _ in range(episodes):
        merged = {a: Q1['A'][a] + Q2['A'][a] for a in actions}
        a = 'left' if rng.random() < epsilon else max(actions, key=lambda x: merged[x])
        if a == 'left':
            left_count += 1
            if rng.random() < 0.5:
                Q1['A']['left'] += alpha * (Q2['B']['go'] - Q1['A']['left'])
                r2 = sample_right_reward(rng)
                Q1['B']['go'] += alpha * (r2 - Q1['B']['go'])
            else:
                Q2['A']['left'] += alpha * (Q1['B']['go'] - Q2['A']['left'])
                r2 = sample_right_reward(rng)
                Q2['B']['go'] += alpha * (r2 - Q2['B']['go'])
        else:
            r = sample_right_reward(rng)
            if rng.random() < 0.5:
                Q1['A']['right'] += alpha * (r - Q1['A']['right'])
            else:
                Q2['A']['right'] += alpha * (r - Q2['A']['right'])
    return left_count / episodes

q_learning(), double_q_learning()

The point of Double Q-learning is not that it always outperforms on every short run, but that separating selection and evaluation reduces systematic overestimation.